In [1]:
# Taylor's Function

import numpy as np
import pandas as pd
#[ERROR] Exception occurred during treelist generation: "None of [Index(['pixel_x', 'pixel_y'], dtype='object')] are in the [columns]"

def generate_treelist(pixel_df, ntrees_50_model, ntrees_16_model, ntrees_84_model, pixel_resolution):
    try:
        pixel_df = pixel_df.rename(columns={'X': 'pixel_x'})
        pixel_df = pixel_df.rename(columns={'Y': 'pixel_y'})
        # Drop rows containing NaN values
        pixel_df = pixel_df.dropna()
        if pixel_df.isnull().values.any():
            print(f"[ERROR] NaNs detected in pixel_df after dropping NaN rows.")
            return None
        print("TESTING 1...")
        # Define the sampleDraws function
        pixel_df = pixel_df.reset_index(drop=True)
        print("TESTING 2...")
        def sampleDraws(df, num_pulls):
            try:
                print("TESTING FUNCTION IN FUNCTION")
                pull_cols = ['pull_' + str(i+1) for i in range(num_pulls)]
                random_samples = np.random.normal(
                    df['mu'].values[:, np.newaxis], df['sd'].values[:, np.newaxis], (len(df), num_pulls))
                samples_df = pd.DataFrame(
                    random_samples, columns=pull_cols, index=df.index)
                out_df = pd.concat([df, samples_df], axis=1)
                return out_df
            except Exception as e:
                print(f"[ERROR] Exception occurred in sampleDraws: {e}")
                return None
        print("TESTING 3...")
        # Main treelist generation code starts here
        pixel_coords = pixel_df[["pixel_x", "pixel_y"]]
        print("TESTING 3.5...")
        pixel_df = pixel_df.drop(
            #columns=['pixel_x', 'pixel_y', '.geo', 'geometry', 'system:index'])
            columns=['pixel_x', 'pixel_y', 'time'])
        print("TESTING 3.75...")
        '''
        desired_columns = [
            'mean_num_of_fires', 'mode_years_since_fire', 'etr_10_mean', 'etr_1_mean',
            'etr_20_mean', 'etr_30_mean', 'etr_5_mean', 'pr_10_mean', 'pr_1_mean',
            'pr_20_mean', 'pr_30_mean', 'pr_5_mean', 'rmax_10_mean', 'rmax_1_mean',
            'rmax_20_mean', 'rmax_30_mean', 'rmax_5_mean', 'rmin_10_mean', 'rmin_1_mean',
            'rmin_20_mean', 'rmin_30_mean', 'rmin_5_mean', 'tmmn_10_mean', 'tmmn_1_mean',
            'tmmn_20_mean', 'tmmn_30_mean', 'tmmn_5_mean', 'tmmx_10_mean', 'tmmx_1_mean',
            'tmmx_20_mean', 'tmmx_30_mean', 'tmmx_5_mean', 'vs_10_mean', 'vs_1_mean',
            'vs_20_mean', 'vs_30_mean', 'vs_5_mean', 'ls_blue', 'ls_green', 'ls_nir',
            'ls_red', 'ls_swir1', 'ls_swir2', 'water_mask'
        ]'''
        
        desired_columns = [
        'tmmx_1_mean', 'tmmn_1_mean', 'pr_1_mean', 'vs_1_mean', 'rmin_1_mean', 
        'rmax_1_mean', 'etr_1_mean', 'tmmx_5_mean', 'tmmn_5_mean', 'pr_5_mean', 
        'vs_5_mean', 'rmin_5_mean', 'rmax_5_mean', 'etr_5_mean', 'tmmx_10_mean', 
        'tmmn_10_mean', 'pr_10_mean', 'vs_10_mean', 'rmin_10_mean', 'rmax_10_mean', 
        'etr_10_mean', 'tmmx_20_mean', 'tmmn_20_mean', 'pr_20_mean', 'vs_20_mean', 
        'rmin_20_mean', 'rmax_20_mean', 'etr_20_mean', 'tmmx_30_mean', 'tmmn_30_mean', 
        'pr_30_mean', 'vs_30_mean', 'rmin_30_mean', 'rmax_30_mean', 'etr_30_mean', 'water_mask', 
        'ls_blue', 'ls_green', 'ls_red', 'ls_nir', 'ls_swir1', 'ls_swir2', 
        'mean_num_of_fires', 'mode_years_since_fire'
        ]

        #print(pixel_df)
        print("TESTING 4...")
        print(pixel_df.columns)
        pixel_df = pixel_df[desired_columns]
        print("TESTING 5...")
        print(f"[INFO] Predicting number of trees...")

        ntrees_50_pred = ntrees_50_model.predict(pixel_df)
        ntrees_16_pred = ntrees_16_model.predict(pixel_df)
        ntrees_84_pred = ntrees_84_model.predict(pixel_df)

        if np.isnan(ntrees_50_pred).any() or np.isnan(ntrees_16_pred).any() or np.isnan(ntrees_84_pred).any():
            print(f"[ERROR] NaNs detected in number of trees prediction.")
            return None

        # Store number of trees' mean and standard deviation for uncertainty
        ntrees_mu = ntrees_50_pred
        ntrees_sd = abs(ntrees_84_pred - ntrees_16_pred) / 2

        df = pd.DataFrame({'mu': ntrees_mu, 'sd': ntrees_sd})

        print(f"[INFO] Sampling number of trees...")
        random_ntrees = sampleDraws(df, num_pulls=1)
        if random_ntrees is None:
            return None
        random_ntrees_clean = random_ntrees.drop(['mu', 'sd'], axis=1)
        


        unscaled_round_ntrees = np.round(random_ntrees_clean).astype(int)
        random_ntrees_clean[random_ntrees_clean < 0] = 0

        current_ntrees = unscaled_round_ntrees.iloc[:, 0]
        current_ntrees[current_ntrees < 0] = 0

        pixel_df = pd.concat([pixel_df, current_ntrees], axis=1)

   
       
        random_coords = np.random.uniform(-pixel_resolution / 2,
                                          pixel_resolution / 2, (len(random_ntrees_clean), 2))
        repeated_pixel_coords = pixel_coords.values.repeat(
            current_ntrees, axis=0)
        tree_coords = repeated_pixel_coords + random_coords

        # Repeat pixel-level variables to match the length of treelist
        ntrees_mu_repeated = np.repeat(ntrees_mu, current_ntrees)
        ntrees_sd_repeated = np.repeat(ntrees_sd, current_ntrees)
      

        # Construct final treelist DataFrame with uncertainty columns
        treelist = pd.DataFrame({
            'pixel_x': repeated_pixel_coords[:, 0],
            'pixel_y': repeated_pixel_coords[:, 1],
            'tree_x': tree_coords[:, 0],
            'tree_y': tree_coords[:, 1],
            'n_trees_mu': ntrees_mu_repeated,
            'n_trees_sd': ntrees_sd_repeated
        })

        print(treelist)

        return treelist

    except Exception as e:
        print(f"[ERROR] Exception occurred during treelist generation: {e}")
        return None

In [3]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()


Successfully saved authorization token.


In [4]:
import ee

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Define the path to the stacked image in Google Cloud Storage
gcs_bucket = 'gigafire_trees1'
stacked_image_path = f'gs://{gcs_bucket}/Stacked_Rasters_2023_vers2_with_mean.tif'

# Load the stacked image
stacked_image = ee.Image.loadGeoTIFF(stacked_image_path).toFloat()
'''
stacked_image_re = stacked_image.rename(
    [
        'tmmx_1_mean', 'tmmn_1_mean', 'pr_1_mean', 'vs_1_mean', 'rmin_1_mean', 
        'rmax_1_mean', 'etr_1_mean', 'tmmx_5_mean', 'tmmn_5_mean', 'pr_5_mean', 
        'vs_5_mean', 'rmin_5_mean', 'rmax_5_mean', 'etr_5_mean', 'tmmx_10_mean', 
        'tmmn_10_mean', 'pr_10_mean', 'vs_10_mean', 'rmin_10_mean', 'rmax_10_mean', 
        'etr_10_mean', 'tmmx_20_mean', 'tmmn_20_mean', 'pr_20_mean', 'vs_20_mean', 
        'rmin_20_mean', 'rmax_20_mean', 'etr_20_mean', 'tmmx_30_mean', 'tmmn_30_mean', 
        'pr_30_mean', 'vs_30_mean', 'rmin_30_mean', 'rmax_30_mean', 'etr_30_mean', 'water_mask', 
        'ls_blue', 'ls_green', 'ls_red', 'ls_nir', 'ls_swir1', 'ls_swir2', 
        'mean_num_of_fires', 'mode_years_since_fire'
    ]
)
stacked_image_reordered = stacked_image_re.select([
            'mean_num_of_fires', 'mode_years_since_fire', 'etr_10_mean', 'etr_1_mean',
            'etr_20_mean', 'etr_30_mean', 'etr_5_mean', 'pr_10_mean', 'pr_1_mean',
            'pr_20_mean', 'pr_30_mean', 'pr_5_mean', 'rmax_10_mean', 'rmax_1_mean',
            'rmax_20_mean', 'rmax_30_mean', 'rmax_5_mean', 'rmin_10_mean', 'rmin_1_mean',
            'rmin_20_mean', 'rmin_30_mean', 'rmin_5_mean', 'tmmn_10_mean', 'tmmn_1_mean',
            'tmmn_20_mean', 'tmmn_30_mean', 'tmmn_5_mean', 'tmmx_10_mean', 'tmmx_1_mean',
            'tmmx_20_mean', 'tmmx_30_mean', 'tmmx_5_mean', 'vs_10_mean', 'vs_1_mean',
            'vs_20_mean', 'vs_30_mean', 'vs_5_mean', 'ls_blue', 'ls_green', 'ls_nir',
            'ls_red', 'ls_swir1', 'ls_swir2', 'water_mask'
        ])
'''
#stacked_image = ee.Image("projects/robust-raster/assets/Stacked_Rasters_2023_vers2")

# Define the Tahoe area (example coordinates for bounding box)
tahoe_area = ee.Geometry.Rectangle([-120.23864293679797, 39.13396055649666, -120.22233510598743, 39.14674208440413])

# THIS WILL GENERATE A FEATURE COLLECTION, WHICH WONT WORK FOR MY PACKAGE! #
# COMMENTING THIS OUT #
'''
# Sample the image within the Tahoe area
sampled_data = stacked_image.sample(
    region=tahoe_area,
    scale=1000,
    projection='EPSG:3310',
    geometries=True
)

sampled_ic = ee.ImageCollection(sampled_data)
'''
ic = ee.ImageCollection(stacked_image)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [15]:
print(ic.first().bandNames().getInfo())

['tmmx_1_mean', 'tmmn_1_mean', 'pr_1_mean', 'vs_1_mean', 'rmin_1_mean', 'rmax_1_mean', 'etr_1_mean', 'tmmx_5_mean', 'tmmn_5_mean', 'pr_5_mean', 'vs_5_mean', 'rmin_5_mean', 'rmax_5_mean', 'etr_5_mean', 'tmmx_10_mean', 'tmmn_10_mean', 'pr_10_mean', 'vs_10_mean', 'rmin_10_mean', 'rmax_10_mean', 'etr_10_mean', 'tmmx_20_mean', 'tmmn_20_mean', 'pr_20_mean', 'vs_20_mean', 'rmin_20_mean', 'rmax_20_mean', 'etr_20_mean', 'tmmx_30_mean', 'tmmn_30_mean', 'pr_30_mean', 'vs_30_mean', 'rmin_30_mean', 'rmax_30_mean', 'etr_30_mean', 'water_mask', 'ls_blue', 'ls_green', 'ls_red', 'ls_nir', 'ls_swir1', 'ls_swir2', 'mean_num_of_fires', 'mode_years_since_fire']


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=tahoe_area, crs='EPSG:3310', scale=1000)

# Drop rows (along dimension 'x') with missing values
filtered_dataset = ds.dropna(dim="X")

# Drop columns (along dimension 'y') with missing values
filtered_dataset = ds.dropna(dim="Y")
print(filtered_dataset)

In [8]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=tahoe_area, crs='EPSG:3310', scale=1000)

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\xee\ext.py:683: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(


In [9]:
print(ds)

<xarray.Dataset> Size: 196B
Dimensions:                (time: 1, X: 1, Y: 1)
Coordinates:
  * time                   (time) int32 4B 0
  * X                      (X) float64 8B -2.011e+04
  * Y                      (Y) float64 8B 1.248e+05
Data variables: (12/44)
    tmmx_1_mean            (time, X, Y) float32 4B ...
    tmmn_1_mean            (time, X, Y) float32 4B ...
    pr_1_mean              (time, X, Y) float32 4B ...
    vs_1_mean              (time, X, Y) float32 4B ...
    rmin_1_mean            (time, X, Y) float32 4B ...
    rmax_1_mean            (time, X, Y) float32 4B ...
    ...                     ...
    ls_red                 (time, X, Y) float32 4B ...
    ls_nir                 (time, X, Y) float32 4B ...
    ls_swir1               (time, X, Y) float32 4B ...
    ls_swir2               (time, X, Y) float32 4B ...
    mean_num_of_fires      (time, X, Y) float32 4B ...
    mode_years_since_fire  (time, X, Y) float32 4B ...
Attributes:
    crs:      EPSG:3310


In [11]:
df = ds.to_dataframe().reset_index()

In [ ]:
print(df)

In [12]:
df.to_csv("TaylorTest_cloud_newarea.csv")

In [ ]:
from joblib import load

ntrees_16_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ntrees_quantile_16_model_watermask.joblib")
ntrees_50_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ntrees_quantile_50_model_watermask.joblib")
ntrees_84_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ntrees_quantile_84_model_watermask.joblib")


# species_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\spcd_classifier_model.joblib")

# dbh_16_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\dbh_quantile_16_model.joblib")
# dbh_50_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\dbh_quantile_50_model.joblib")
# dbh_84_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\dbh_quantile_84_model.joblib")

# ht_16_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ht_quantile_16_model.joblib")
# ht_50_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ht_quantile_50_model.joblib")
# ht_84_model = load(r"R:\Users\adrianom.UNR\Downloads\adriano_chpt1_synthetic_treelist_files\adriano_chpt1_synthetic_treelist_files\ht_quantile_84_model.joblib")


In [17]:
pixel_resolution = 1000

generate_treelist(df, ntrees_50_model, ntrees_16_model, ntrees_84_model, pixel_resolution)

TESTING 1...
TESTING 2...
TESTING 3...
TESTING 3.5...
TESTING 3.75...
TESTING 4...
Index(['tmmx_1_mean', 'tmmn_1_mean', 'pr_1_mean', 'vs_1_mean', 'rmin_1_mean',
       'rmax_1_mean', 'etr_1_mean', 'tmmx_5_mean', 'tmmn_5_mean', 'pr_5_mean',
       'vs_5_mean', 'rmin_5_mean', 'rmax_5_mean', 'etr_5_mean', 'tmmx_10_mean',
       'tmmn_10_mean', 'pr_10_mean', 'vs_10_mean', 'rmin_10_mean',
       'rmax_10_mean', 'etr_10_mean', 'tmmx_20_mean', 'tmmn_20_mean',
       'pr_20_mean', 'vs_20_mean', 'rmin_20_mean', 'rmax_20_mean',
       'etr_20_mean', 'tmmx_30_mean', 'tmmn_30_mean', 'pr_30_mean',
       'vs_30_mean', 'rmin_30_mean', 'rmax_30_mean', 'etr_30_mean',
       'water_mask', 'ls_blue', 'ls_green', 'ls_red', 'ls_nir', 'ls_swir1',
       'ls_swir2', 'mean_num_of_fires', 'mode_years_since_fire'],
      dtype='object')
TESTING 5...
[INFO] Predicting number of trees...
[ERROR] Exception occurred during treelist generation: The feature names should match those that were passed during fit.
Featu

In [5]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run

pixel_resolution = 1000

#generate_treelist(pixel_df, ntrees_50_model, ntrees_16_model, ntrees_84_model, species_model, dbh_50_model, dbh_16_model, dbh_84_model, ht_50_model, ht_16_model, ht_84_model, pixel_resolution)

treelist = pd.DataFrame(columns=["pixel_x", "pixel_y", "tree_x", "tree_y", "n_trees_mu", "n_trees_sd"])

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': tahoe_area,
    'crs': 'EPSG:3310',
    'scale': 1000
    },
    user_function=generate_treelist,
    output_template=treelist,
    user_function_kwargs={
        "ntrees_50_model": ntrees_50_model, 
        "ntrees_16_model": ntrees_16_model, 
        "ntrees_84_model": ntrees_84_model, 
        "pixel_resolution": pixel_resolution
        },
    export_kwargs={
        "mode": "vector",
        "file_format": "CSV",
        "output_folder": "tiles33", 
        #"export_to_gcs": True,
        #"gcs_credentials": r"R:\SCRATCH\adrianom\credentials\cloud_storage\robust-raster-b9a2bc301bc9.json",
        #"gcs_bucket": "raster-bucket-test25678"},
        },
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 4,
        "threads_per_worker": 1,
        "memory_limit": "1GB"
    }
)

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\google\cloud\storage\_http.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


NameError: name 'ntrees_50_model' is not defined